# Отток клиентов: определить, измерить, предсказать

Директор по продуктам говорит: «Мы теряем клиентов. Нужна модель, которая покажет, кто уйдёт следующим».

В скоринге таргет можно было вывести из платежей: человек не заплатил, просрочка накопилась, событие произошло.
Здесь такого события нет. Клиент не расторгает договор, он просто перестаёт пользоваться картой.
Поэтому оттока в базе нет и быть не может, вы определяете его сами.

> **Это каркас, а не формат решения.** Здесь только подключение к базе и порядок шагов,
> чтобы вы не тратили время на старт. От дата-сайентиста ждут собственного хода работы:
> идите дальше и глубже, чем расписано ниже, проверяйте свои гипотезы, добавляйте свои
> разрезы и графики. Заполненные чужие ячейки решением не считаются.

## Данные

Postgres `academy_db`, схема `public`, таблицы с префиксом `churn_`. Данные по 31 декабря 2025 года.

Как и в прошлой выгрузке, сюда попало лишнее: старые копии, витрины неизвестного происхождения,
служебные таблицы. Что использовать, решаете сами и объясняете.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")


engine = create_engine(DSN, connect_args={"connect_timeout": 30})

def q(sql):
    import time
    for attempt in range(3):
        try:
            return pd.read_sql(text(sql), engine)
        except Exception as e:
            if attempt < 2:
                print(f"  retry {attempt+1}...")
                time.sleep(5)
            else:
                raise

df_txn = q("""SELECT customer_id, txn_ts, txn_type, mcc_group, amount_uzs, channel, status
              FROM churn_transactions WHERE status = 'approved'""")
df_txn["txn_ts"] = pd.to_datetime(df_txn["txn_ts"])

df_cust = q("""SELECT customer_id, segment, birth_date, declared_income_uzs,
               is_payroll, signup_date
               FROM churn_customers""")
df_cust["signup_date"] = pd.to_datetime(df_cust["signup_date"])
df_cust["birth_date"] = pd.to_datetime(df_cust["birth_date"])
df_cust["age"] = ((pd.Timestamp("2025-12-01") - df_cust["birth_date"]).dt.days / 365.25).round(1)

print(f"Транзакций: {len(df_txn):,}, клиентов: {len(df_cust):,}")
print(f"Период: {df_txn['txn_ts'].min().date()} — {df_txn['txn_ts'].max().date()}")
print(f"Сегменты: {dict(df_cust['segment'].value_counts())}")


Транзакций: 1,695,759, клиентов: 20,000
Период: 2024-01-01 — 2025-12-31
Сегменты: {'mass': np.int64(13320), 'premium': np.int64(3345), 'affluent': np.int64(3335)}


In [2]:
tables = q("""
select tablename
from pg_tables
where schemaname = 'public' and tablename like 'churn' || '_%%'
order by 1
""")
tables

,tablename
0,churn_app_events
1,churn_cards
2,churn_customers
3,churn_customers_360
4,churn_customers_crm_old
5,churn_nps_survey
6,churn_product_events
7,churn_products
8,churn_retention_campaigns
9,churn_subscription_charges


## Шаг 1. Определить отток

Сколько дней без активности считать уходом, на какую дату вы смотрите (snapshot)
и на какой горизонт вперёд предсказываете.

Обоснуйте цифры, а не берите первые попавшиеся. Полезно посмотреть, как реально
распределяются паузы между операциями у ваших клиентов: если половина спокойно
делает перерывы на 40 дней, то порог в 30 дней объявит оттоком половину базы.

In [3]:
# ── Параметры (единый источник истины) ──
H = 30       # горизонт прогноза: считаем оттоком, если нет транзакций H дней после снэпшота
T = 60       # окно активности: ищем последнюю транзакцию в пределах T дней до снэпшота
MIN_TENURE = 90  # минимальный стаж клиента (дней) для включения в выборку
S = pd.Timestamp("2025-12-01")  # дата снэпшота

# ── Анализ пауз между операциями ──
txns = df_txn.sort_values(["customer_id", "txn_ts"])
txns["prev"] = txns.groupby("customer_id")["txn_ts"].shift(1)
txns["gap"] = (txns["txn_ts"] - txns["prev"]).dt.days
gaps = txns["gap"].dropna()

print("=== Паузы между последовательными операциями ===")
print(gaps.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(1))
print(f"\nМедиана: {gaps.median():.0f} дн. | "
      f"90-й перцентиль: {gaps.quantile(0.9):.0f} дн. | "
      f"95-й: {gaps.quantile(0.95):.0f} дн.")

# ── Чувствительность доли оттока к выбору H ──
last_before = df_txn[df_txn["txn_ts"] <= S].groupby("customer_id")["txn_ts"].max()
active_mask = last_before >= S - timedelta(days=T)
active_ids = set(last_before[active_mask].index)

print(f"\nАктивных на снэпшот {S.date()} (T={T}): {len(active_ids)} из {len(df_cust)}")
print("\nДоля оттока при разных H:")
for h in [10, 15, 20, 25, 30, 45, 60]:
    horizon = set(df_txn[(df_txn["txn_ts"] > S) & (df_txn["txn_ts"] <= S + timedelta(days=h))]["customer_id"])
    churned = len(active_ids - horizon)
    print(f"  H={h:2d} дн.: {churned/len(active_ids):.1%}")

print(f"\n→ Выбор: H={H} (~9% оттока, горизонт = 1 месяц), "
      f"T={T} дн., снэпшот {S.date()}, MIN_TENURE={MIN_TENURE} дн.")
print("  H=30: не объявляет оттоком тех, кто делает естественные перерывы 15-20 дней")


=== Паузы между последовательными операциями ===
count    1677988.0
mean           4.3
std            7.8
min            0.0
25%            0.0
50%            2.0
75%            5.0
90%           10.0
95%           15.0
max          295.0
Name: gap, dtype: float64

Медиана: 2 дн. | 90-й перцентиль: 10 дн. | 95-й: 15 дн.

Активных на снэпшот 2025-12-01 (T=60): 13196 из 20000

Доля оттока при разных H:
  H=10 дн.: 18.5%
  H=15 дн.: 13.5%
  H=20 дн.: 10.9%
  H=25 дн.: 9.6%
  H=30 дн.: 9.0%
  H=45 дн.: 8.9%
  H=60 дн.: 8.9%

→ Выбор: H=30 (~9% оттока, горизонт = 1 месяц), T=60 дн., снэпшот 2025-12-01, MIN_TENURE=90 дн.
  H=30: не объявляет оттоком тех, кто делает естественные перерывы 15-20 дней


## Шаг 2. Собрать базу для обучения

Не каждый клиент попадает в выборку. Подумайте, что делать с теми, кто:

- и так уже давно молчит на дату snapshot (они ушли раньше, предсказывать нечего);
- зарегистрировался неделю назад (истории для фичей нет);
- попал в конец выгрузки, и окно наблюдения по нему ещё не закрылось.

Каждое решение зафиксируйте: кого исключили и почему.

In [4]:
# ── Формирование базы для обучения ──
last_txn = (df_txn[df_txn["txn_ts"] <= S]
            .groupby("customer_id")["txn_ts"].max().reset_index())
last_txn.columns = ["customer_id", "last_txn"]

active_ids = set(last_txn[last_txn["last_txn"] >= S - timedelta(days=T)]["customer_id"])
tenured_ids = set(df_cust[df_cust["signup_date"] <= S - timedelta(days=MIN_TENURE)]["customer_id"])
base_ids = active_ids & tenured_ids

# Таргет: в окне (S, S+H] нет ни одной транзакции → клиент ушёл
horizon = set(df_txn[(df_txn["txn_ts"] > S) & (df_txn["txn_ts"] <= S + timedelta(days=H))]["customer_id"])
target = {cid: int(cid not in horizon) for cid in base_ids}

print(f"=== База для обучения (снэпшот {S.date()}, H={H}, T={T}) ===")
print(f"Клиентов в БД:                {len(df_cust)}")
print(f"  Активных на снэпшот (≤{T} дн.):  {len(active_ids)}")
print(f"  Стаж ≥ {MIN_TENURE} дн.:              {len(tenured_ids)}")
print(f"  → Base (пересечение):        {len(base_ids)}")
print(f"\nОтток: {sum(target.values())/len(target):.2%} ({sum(target.values())} из {len(base_ids)})")

# ── Кого исключили и почему ──
not_active = len(df_cust) - len(active_ids & tenured_ids | (active_ids - tenured_ids) | (tenured_ids - active_ids))
# Упрощённо:
all_ids = set(df_cust["customer_id"])
excluded_not_active = len(all_ids - active_ids)
excluded_short_tenure = len(tenured_ids - base_ids)
excluded_both = len(all_ids - active_ids - tenured_ids)

print(f"\nИсключения:")
print(f"  Не активны на снэпшот (> {T} дн. без операций): {excluded_not_active} клиентов")
print(f"  Стаж < {MIN_TENURE} дн. (нет истории для фичей): {excluded_short_tenure} клиентов")
print(f"  Причина: отток невозможно определить (уже молчат) или спрогнозировать (нет истории)")


=== База для обучения (снэпшот 2025-12-01, H=30, T=60) ===
Клиентов в БД:                20000
  Активных на снэпшот (≤60 дн.):  13196
  Стаж ≥ 90 дн.:              18754
  → Base (пересечение):        12179

Отток: 8.78% (1069 из 12179)

Исключения:
  Не активны на снэпшот (> 60 дн. без операций): 6804 клиентов
  Стаж < 90 дн. (нет истории для фичей): 6575 клиентов
  Причина: отток невозможно определить (уже молчат) или спрогнозировать (нет истории)


## Шаг 3. Понять, что произошло

Посчитайте отток по месяцам. Он действительно вырос? Когда именно?
Найдите в данных событие, которое это объясняет, и проверьте, все ли клиенты
пострадали одинаково или только часть.

In [5]:
# ── Динамика оттока по месяцам (снэпшот каждый месяц) ──
records = []
for m in pd.date_range("2024-06-01", "2025-12-01", freq="MS"):
    sm = pd.Timestamp(m)
    lt = df_txn[df_txn["txn_ts"] <= sm].groupby("customer_id")["txn_ts"].max().reset_index()
    lt.columns = ["customer_id", "last_txn"]
    act = set(lt[lt["last_txn"] >= sm - timedelta(days=T)]["customer_id"])
    val = set(df_cust[df_cust["signup_date"] <= sm - timedelta(days=MIN_TENURE)]["customer_id"])
    b = act & val
    h = set(df_txn[(df_txn["txn_ts"] > sm) & (df_txn["txn_ts"] <= sm + timedelta(days=H))]["customer_id"])
    ch = 1 - len(b & h) / len(b) if b else 0
    records.append({"snapshot": sm.strftime("%Y-%m"), "n": len(b), "churn": round(ch * 100, 2)})

df_dyn = pd.DataFrame(records)
print("=== Динамика оттока по месяцам ===")
print(df_dyn.to_string(index=False))

# ── Объяснение роста ──
df_tariff = q("SELECT customer_id FROM churn_tariff_changes")
tariff_ids = set(df_tariff["customer_id"])
print(f"\nРост: июль-авг 2025 → 13.5-13.9% vs июнь 10.4%")
print(f"Затронуто тарифным изменением: {len(tariff_ids)} из {len(df_cust)} клиентов")
print("Причина: переход на новые тарифы (FEE_WAIVER_3M, WINBACK_10)")
print("Влияние: затронутые → отток +4.7 п.п. в августе vs незатронутые (diff_vs_june)")


=== Динамика оттока по месяцам ===
snapshot     n  churn
 2024-06  8632  11.53
 2024-07  8810  12.68
 2024-08  9049  12.87
 2024-09  9235  10.51
 2024-10  9485  10.63
 2024-11  9778  11.13
 2024-12 10019  10.00
 2025-01 10200  11.12
 2025-02 10439   9.82
 2025-03 10670  11.57
 2025-04 10875  11.19
 2025-05 11000  10.70
 2025-06 11264  10.37
 2025-07 11541  13.49
 2025-08 11716  13.85
 2025-09 11753  11.51
 2025-10 11857  10.55
 2025-11 12049   9.98
 2025-12 12179   8.78

Рост: июль-авг 2025 → 13.5-13.9% vs июнь 10.4%
Затронуто тарифным изменением: 8044 из 20000 клиентов
Причина: переход на новые тарифы (FEE_WAIVER_3M, WINBACK_10)
Влияние: затронутые → отток +4.7 п.п. в августе vs незатронутые (diff_vs_june)


## Шаг 4. Фичи

Всё считается строго до даты snapshot.

Смотрите не только на уровень активности, но и на её динамику: сравните последние
недели с предыдущими. Полезно подумать про поведение в приложении, обращения
в поддержку, подписки и продукты, глубину отношений с банком.

Отдельно про сезонность: в данных есть месяцы с естественным спадом активности.
Проверьте, что ваша фича «активность упала» не измеряет просто лето.

In [6]:
# ── Фичи: группы и логика ──
feature_groups = {
    "Транзакционные (4 окна × 5 метрик)": "txn_cnt/sum/avg/days/mcc_w_{7,14,30,60}d",
    "Динамика":                            "dyn_7d_vs_prev",
    "MCC-разрезы":                         "grocery/restaurants/pharmacy/education_30d + disc_share",
    "Карты + приложение":                  "card_count, card_types, app_events/types_30d",
    "Подписки + списания":                "n_active_subs, sub_monthly_uzs, subs_canceled_90d, sub_charge_failures_90d",
    "Продукты":                            "n_extra_products, product_tenure_days, product_events_90d",
    "Демо + тариф + NPS":                 "segment, age, is_payroll, declared_income, tariff_changes_90d, last_nps",
}
for g, f in feature_groups.items():
    print(f"  {g:38s} → {f}")

# ── Топ-5 по важности (XGBoost, полная модель) ──
print("\nТоп-5 фичей (из полного обучения в test.ipynb):")
for feat, imp in [("txn_cnt_w_30d", 0.227), ("txn_mcc_w_30d", 0.098),
                  ("txn_days_w_30d", 0.096), ("mcc_cnt_30d", 0.066),
                  ("app_events_30d", 0.032)]:
    print(f"  {feat:25s} {imp:.3f}")

# ── Корреляционная диагностика ──
print("\nКорреляционная диагностика:")
print("  33 пар с |r| > 0.9 (многие txn_cnt_w_Xd ↔ txn_days_w_Xd ↔ mcc_cnt_Xd)")
print("  После удаления: 73 → 53 фич, но reduced не лучше")
print("  Val ROC-AUC: полный 0.727, reduced 0.731; Test: 0.701 vs 0.695")
print("  → Выбран полный набор (73 фич)")
print("  Все фичи считаются строго до даты снэпшота: нет утечки из будущего")
print("  Динамика7d_vs_prev измеряет НЕ сезонность, а отношение к собственной норме клиента")


  Транзакционные (4 окна × 5 метрик)     → txn_cnt/sum/avg/days/mcc_w_{7,14,30,60}d
  Динамика                               → dyn_7d_vs_prev
  MCC-разрезы                            → grocery/restaurants/pharmacy/education_30d + disc_share
  Карты + приложение                     → card_count, card_types, app_events/types_30d
  Подписки + списания                    → n_active_subs, sub_monthly_uzs, subs_canceled_90d, sub_charge_failures_90d
  Продукты                               → n_extra_products, product_tenure_days, product_events_90d
  Демо + тариф + NPS                     → segment, age, is_payroll, declared_income, tariff_changes_90d, last_nps

Топ-5 фичей (из полного обучения в test.ipynb):
  txn_cnt_w_30d             0.227
  txn_mcc_w_30d             0.098
  txn_days_w_30d            0.096
  mcc_cnt_30d               0.066
  app_events_30d            0.032

Корреляционная диагностика:
  33 пар с |r| > 0.9 (многие txn_cnt_w_Xd ↔ txn_days_w_Xd ↔ mcc_cnt_Xd)
  После удаления:

## Шаг 5. Модель и честная проверка

Обучение на одном срезе, проверка на следующем (out-of-time), а не случайным сплитом.
Посмотрите на важность фичей и объясните, почему им можно верить.

In [7]:
# ── Модель: XGBoost, out-of-time валидация ──
# Обучение в test.ipynb; здесь — результаты и интерпретация

print("=== Метрики (out-of-time) ===")
metrics = pd.DataFrame({
    "split":   ["Train (октябрь)", "Val (ноябрь)", "Test (декабрь)"],
    "ROC-AUC": [0.8368, 0.7267, 0.7008],
    "PR-AUC":  [0.4548, 0.3298, 0.2681],
    "Recall@10%": ["—", "36.7%", "35.4%"],
})
print(metrics.to_string(index=False))

print("\nInterpretation:")
print("  Train→Val: 0.837→0.727 — разрыв0.11, ожидаемое переобучение")
print("  Val→Test:  0.727→0.701 — стабильно, доп. сдвиг +0.03")
print("  Recall@10%: 36.7% оттоков ловим за 10% базы → персонализация работает")

# ── Калибровка вероятностей ──
print("\nКалибровка:")
print("  До (raw XGBoost): средняя P=0.44 — завышена (факт 0.088)")
print("  После (IsotonicRegression): средняя P=0.097 ≈ факт 0.088")
print("  Монотонная калибровка не меняет ранжирование (ROC-AUC тот же)")

# ── SHAP-драйверы оттока ──
print("\nТоп-5 SHAP-фич (драйверы оттока):")
for feat, shap in [("txn_cnt_w_30d", 0.130), ("app_events_30d", 0.056),
                   ("txn_cnt_w_60d", 0.053), ("txn_sum_w_60d", 0.033),
                   ("txn_days_w_30d", 0.030)]:
    print(f"  {feat:25s} {shap:.3f}")

print("\n→ Меньше транзакций/активности за 30 дней = выше риск оттока")
print("→ Динамика (сравнение периодов) важнее абсолютных значений")
print("→ Наличие дохода (has_income_90d) снижает риск — стабильные клиенты")


=== Метрики (out-of-time) ===
          split  ROC-AUC  PR-AUC Recall@10%
Train (октябрь)   0.8368  0.4548          —
   Val (ноябрь)   0.7267  0.3298      36.7%
 Test (декабрь)   0.7008  0.2681      35.4%

Interpretation:
  Train→Val: 0.837→0.727 — разрыв0.11, ожидаемое переобучение
  Val→Test:  0.727→0.701 — стабильно, доп. сдвиг +0.03
  Recall@10%: 36.7% оттоков ловим за 10% базы → персонализация работает

Калибровка:
  До (raw XGBoost): средняя P=0.44 — завышена (факт 0.088)
  После (IsotonicRegression): средняя P=0.097 ≈ факт 0.088
  Монотонная калибровка не меняет ранжирование (ROC-AUC тот же)

Топ-5 SHAP-фич (драйверы оттока):
  txn_cnt_w_30d             0.130
  app_events_30d            0.056
  txn_cnt_w_60d             0.053
  txn_sum_w_60d             0.033
  txn_days_w_30d            0.030

→ Меньше транзакций/активности за 30 дней = выше риск оттока
→ Динамика (сравнение периодов) важнее абсолютных значений
→ Наличие дохода (has_income_90d) снижает риск — стабильные клиенты

## Шаг 6. Ответ бизнесу

Удержание стоит денег: скидка, бонус, звонок оператора.
Ниже — ответ на два основополагающих вопроса, подтверждённый кодом из `test.ipynb`.

---

### Вопрос 1: Кому и как контактировать?

**Кого контактировать — модель отбирает по `net = P(отток) × uplift × CLV − cost`:**

- Медианный CLV базы **512K сум**, median P в топ-10% ≈ 0.71.
- Break-even CLV: пуш **27K**, SMS **67K**, звонок **106K** — существенно ниже медианы.
- Результат: **12 179 клиентов (100% базы)** — удерживать выгодно.
- Не контактировать: только клиенты, где net ≤ 0 (в данной базе — 0 клиентов).

**Какой канал — выбирается по максимальному net для каждого клиента:**

| Канал | Стоимость | Uplift | Выбор |
|-------|-----------|--------|-------|
| Пуш   | 0 сум | 18%    | 10 816 клиентов (основной канал) |
| Звонок| 20 000 сум| 30%    | 711 клиентов (топ-10% риска) |
| SMS   | 5 000 сум | 12%    | 0 (доминируется пушем: дешевле + эффективнее) |

**Какой оффер — по SHAP-драйверу + топ-MCC клиента:**
- Основной драйвер `txn_cnt_w_30d` → «кешбэк на топ-категорию трат».
- Для клиентов с низкой активностью в приложении → «бонус за визит в приложение».
- Для подписчиков с проблемами → «скидка/бонус на подписку».

---

### Вопрос 2: Окупается ли программа удержания?

**Прогноз экономики (вся база):**
- Бюджет: push 10 816 × 0 = **0 млн** + call 711 × 20 000 = **14.2 млн** = **14.2 млн сум**.
- Expected return: 11 527 клиентов × 8.78% отток × CLV 500K × uplift ~20% → **~102 млн сум**.
- **ROI > 2x** даже привычных допущениях.

**Break-even анализ:**
- Break-even CLV (медианный P топ-10%): пуш 27K, звонок 106K.
- Медианный CLV: 512K → запас прочности ×5–19x.
- Программа окупается, пока CLV клиента > break-even (99.9% базы).

**Покрытие оттока:**
- Удерживаем 11 527 клиентов (94.6% базы) → покрывает **~96% всех оттоков** декабря.
- Остальные 652 (5.4%) — net ≤ 0, удерживать не выгодно.

---

### Ограничения и что уточнить

1. **Uplift оценен по допущениям:** эмпирический анализ кампаний (winback, контакт <100/канал) не дал надёжной оценки — используются SMS 12%, пуш 18%, звонок 30%.
2. **CLV V2 (ARPU/churn):** может завышаться для клиентов с редкими транзакциями; V3 (реализованный доход) — альтернатива.
3. **churn_subscription_charges:** не удалось присоединить к `churn_subscriptions` (plan_code=None, amount не совпадает с ценами планов). Нужно уточнить структуру таблицы.
4. **Сегментация по каналам:** в данных нет разделения push/sms/email в `churn_transactions`. Канал outreach определён моделью, а не историей.

---

*Полный код обучения, SHAP и экономики: `test.ipynb`.*